# Chapter 27 Lab: Final Project Studio

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-27-final-project-studio-lab.ipynb)

This lab is a **project studio**. It gives you a reusable workflow for a final project in modern time series analysis and sequence learning.

The goal is not to learn one more model. The goal is to learn how to organize a complete project:

1. define the forecasting or sequence-learning task clearly,
2. audit and visualize the data,
3. create leakage-safe features,
4. compare baselines and machine-learning models,
5. evaluate by chronological validation and rolling-origin validation,
6. quantify uncertainty,
7. write a reproducible model card and final report.

You can run the notebook first with the simulated data. Then replace the simulated data with your own project data.

## 0. Independent-study checklist

By the end of this lab, you should be able to answer these questions for your own project.

- What is the target variable?
- At what time is each prediction made?
- What information is available at prediction time?
- What is the forecast horizon?
- Which simple baseline must your model beat?
- Which validation design respects time ordering?
- Which metric matches the project goal?
- What uncertainty statement accompanies the point forecast?
- What would make the result unreliable or non-reproducible?

Keep these questions visible while working on your final project.

## 1. Imports and reproducibility

We use standard Colab-friendly packages. The notebook avoids external data downloads so that it is reproducible immediately.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 30)

## 2. Project contract

A strong time-series project begins with a contract. The contract prevents vague modeling and prevents leakage.

For a forecasting problem, a minimal contract contains:

- **unit of observation:** one row per day, hour, patient visit, customer, sensor cycle, etc.;
- **target:** the variable to predict;
- **prediction time:** when the model is allowed to make the prediction;
- **information set:** columns known at that time;
- **horizon:** how far into the future the prediction is;
- **baseline:** a simple method the model must beat;
- **validation:** chronological split or rolling-origin validation;
- **metric:** the loss used for model comparison.

The dictionary below is a template. Replace it with your own project contract when you use this lab for your final project.

In [ ]:
project_contract = {
    "working_title": "Daily demand forecasting project studio",
    "unit_of_observation": "one row per calendar day",
    "target": "demand",
    "prediction_time": "end of day t, before observing day t+1",
    "forecast_horizon": "1 to 14 days ahead",
    "information_set": [
        "past demand values",
        "calendar variables known in advance",
        "planned promotion indicator known in advance",
        "price known or planned in advance"
    ],
    "forbidden_information": [
        "future demand",
        "rolling means computed using future values",
        "scalers fit on validation or test rows"
    ],
    "baselines": ["persistence", "seasonal naive", "moving average"],
    "candidate_models": ["ridge regression", "random forest", "gradient boosting"],
    "validation_design": "chronological split plus rolling-origin validation",
    "primary_metric": "MAE",
    "secondary_metrics": ["RMSE", "MASE", "coverage of prediction intervals"]
}
project_contract

## 3. Simulate a realistic project data set

The project data set mimics daily demand. It has trend, weekly seasonality, annual seasonality, promotions, price effects, temperature effects, missing values, and outliers.

In your own project, this is the section you replace with a data-loading section such as `pd.read_csv(...)`.

In [ ]:
def simulate_daily_project_data(n_days=3 * 365, seed=7339):
    local_rng = np.random.default_rng(seed)
    dates = pd.date_range("2023-01-01", periods=n_days, freq="D")
    t = np.arange(n_days)
    dow = dates.dayofweek.to_numpy()
    month = dates.month.to_numpy()

    trend = 0.035 * t
    weekly = 8.0 * np.sin(2 * np.pi * dow / 7 - 0.7)
    annual = 10.0 * np.sin(2 * np.pi * t / 365.25 - 0.4)

    promotion = ((dow == 4) & (local_rng.random(n_days) < 0.45)).astype(int)
    holiday = (((month == 11) & (dates.day >= 20)) | ((month == 12) & (dates.day <= 31))).astype(int)

    temperature = 62 + 18 * np.sin(2 * np.pi * t / 365.25 - 1.2) + local_rng.normal(0, 4, size=n_days)
    price = 12.0 - 0.18 * promotion + 0.12 * np.sin(2 * np.pi * t / 30) + local_rng.normal(0, 0.08, size=n_days)

    noise_scale = 2.5 + 1.2 * holiday
    noise = local_rng.normal(0, noise_scale, size=n_days)

    demand = (
        90
        + trend
        + weekly
        + annual
        + 7.5 * promotion
        + 5.0 * holiday
        - 4.0 * (price - price.mean())
        + 0.08 * (temperature - temperature.mean())
        + noise
    )

    observed = demand.copy()
    missing_idx = local_rng.choice(n_days, size=14, replace=False)
    outlier_idx = local_rng.choice(np.setdiff1d(np.arange(n_days), missing_idx), size=8, replace=False)
    observed[missing_idx] = np.nan
    observed[outlier_idx] += local_rng.choice([-1, 1], size=len(outlier_idx)) * local_rng.uniform(18, 30, size=len(outlier_idx))

    return pd.DataFrame({
        "ds": dates,
        "t": t,
        "dow": dow,
        "month": month,
        "promotion": promotion,
        "holiday": holiday,
        "temperature": temperature,
        "price": price,
        "demand_true": demand,
        "demand": observed,
        "is_missing_injected": np.isnan(observed),
        "is_outlier_injected": np.isin(np.arange(n_days), outlier_idx)
    })

df_raw = simulate_daily_project_data()
print(df_raw.shape)
df_raw.head()

## 4. Data audit

A project report should include a small data audit before modeling. The audit should answer:

- How many rows and columns are present?
- What is the time range?
- Are there missing observations?
- Are there suspicious outliers?
- Are exogenous variables available at prediction time?

In [ ]:
def data_audit(df, time_col="ds", target_col="demand"):
    audit = {
        "n_rows": len(df),
        "n_columns": df.shape[1],
        "start_time": df[time_col].min(),
        "end_time": df[time_col].max(),
        "missing_target_count": int(df[target_col].isna().sum()),
        "duplicate_time_count": int(df[time_col].duplicated().sum()),
        "target_min": float(df[target_col].min(skipna=True)),
        "target_max": float(df[target_col].max(skipna=True)),
        "target_mean": float(df[target_col].mean(skipna=True))
    }
    return pd.Series(audit)

data_audit(df_raw)

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(df_raw["ds"], df_raw["demand"], linewidth=1, label="observed demand")
plt.scatter(df_raw.loc[df_raw["is_missing_injected"], "ds"],
            np.repeat(df_raw["demand"].median(skipna=True), df_raw["is_missing_injected"].sum()),
            marker="x", label="injected missing")
plt.scatter(df_raw.loc[df_raw["is_outlier_injected"], "ds"],
            df_raw.loc[df_raw["is_outlier_injected"], "demand"],
            marker="o", label="injected outlier")
plt.title("Project data audit plot")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 5. Clean the target without using future information

For this studio example, we use a simple training-only median imputation procedure for missing target values. In a real project, document every cleaning decision.

Outlier handling is not automatic here. Instead, we flag suspicious values and ask whether they are errors or real extreme events.

In [ ]:
def robust_z_score(x):
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad == 0:
        return np.zeros_like(x, dtype=float)
    return 0.6745 * (x - med) / mad

# For plotting and lag construction, fill missing demand by a forward fill followed by median fill.
df = df_raw.copy()
df["demand_clean"] = df["demand"].ffill().fillna(df["demand"].median())
df["robust_z"] = robust_z_score(df["demand_clean"].to_numpy())
df["suspicious_target"] = np.abs(df["robust_z"]) > 4

print("Missing after cleaning:", int(df["demand_clean"].isna().sum()))
print("Suspicious target count:", int(df["suspicious_target"].sum()))
df.loc[df["suspicious_target"], ["ds", "demand", "demand_clean", "robust_z"]].head()

## 6. Chronological splits

A random split is usually not appropriate for time series. We split the observations in chronological order.

The last block is the test set. The block before that is the validation set. Everything earlier is the training set.

In [ ]:
def chronological_split(df, valid_size=120, test_size=120):
    n = len(df)
    train_end = n - valid_size - test_size
    valid_end = n - test_size
    split = np.array(["train"] * n, dtype=object)
    split[train_end:valid_end] = "valid"
    split[valid_end:] = "test"
    return split

df["split"] = chronological_split(df)
df["split"].value_counts(sort=False)

In [ ]:
plt.figure(figsize=(10, 3.5))
for split_name in ["train", "valid", "test"]:
    part = df[df["split"] == split_name]
    plt.plot(part["ds"], part["demand_clean"], linewidth=1, label=split_name)
plt.title("Chronological train/validation/test split")
plt.xlabel("date")
plt.ylabel("cleaned demand")
plt.legend()
plt.show()

## 7. Leakage-safe feature factory

A feature factory is a function that creates all model inputs. This is better than scattered feature code because it makes the project easier to audit.

The key rule is: features in row `t` must only use information available at prediction time.

For one-step-ahead forecasting, lag 1 is allowed, lag 7 is allowed, and calendar variables for tomorrow may be allowed if the calendar is known. A centered rolling mean using future values is not allowed.

In [ ]:
def add_time_series_features(data, target_col="demand_clean"):
    out = data.copy()

    # Lag features use the past only.
    for lag in [1, 2, 3, 7, 14, 28]:
        out[f"lag_{lag}"] = out[target_col].shift(lag)

    # Rolling features are shifted first so that row t does not use y_t.
    shifted = out[target_col].shift(1)
    for window in [7, 14, 28]:
        out[f"roll_mean_{window}"] = shifted.rolling(window).mean()
        out[f"roll_std_{window}"] = shifted.rolling(window).std()

    # Calendar features are known in advance.
    out["dow_sin"] = np.sin(2 * np.pi * out["dow"] / 7)
    out["dow_cos"] = np.cos(2 * np.pi * out["dow"] / 7)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)

    # Fourier features for annual seasonality.
    for k in [1, 2]:
        out[f"annual_sin_{k}"] = np.sin(2 * np.pi * k * out["t"] / 365.25)
        out[f"annual_cos_{k}"] = np.cos(2 * np.pi * k * out["t"] / 365.25)

    return out

feature_df = add_time_series_features(df)
feature_cols = [
    c for c in feature_df.columns
    if c.startswith("lag_") or c.startswith("roll_") or c.endswith("_sin") or c.endswith("_cos")
] + ["promotion", "holiday", "temperature", "price"]

feature_df[feature_cols + ["demand_clean", "split"]].head(10)

## 8. Target alignment unit tests

A final project should include simple tests that verify the feature rows and target rows mean what you claim.

The next tests check that `lag_1` at row `t` equals the previous target and that `roll_mean_7` at row `t` averages the previous seven targets, not the current one.

In [ ]:
def test_feature_alignment(feature_df, target_col="demand_clean"):
    check_rows = [30, 100, 300]
    for i in check_rows:
        observed_lag_1 = feature_df.loc[i, "lag_1"]
        expected_lag_1 = feature_df.loc[i - 1, target_col]
        assert np.isclose(observed_lag_1, expected_lag_1), f"lag_1 failed at row {i}"

        observed_roll = feature_df.loc[i, "roll_mean_7"]
        expected_roll = feature_df.loc[i - 7:i - 1, target_col].mean()
        assert np.isclose(observed_roll, expected_roll), f"roll_mean_7 failed at row {i}"
    return "All alignment tests passed."

test_feature_alignment(feature_df)

## 9. Baselines

A baseline is a serious model. It gives the minimum standard your more complex model must beat.

We evaluate:

- **persistence:** tomorrow equals today;
- **seasonal naive:** tomorrow equals the value from seven days ago;
- **moving average:** tomorrow equals the average of the previous seven days.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mase(y_true, y_pred, y_train, seasonal_period=7):
    denom = np.mean(np.abs(y_train[seasonal_period:] - y_train[:-seasonal_period]))
    return float(np.mean(np.abs(y_true - y_pred)) / denom)

def metric_row(name, y_true, y_pred, y_train):
    return {
        "model": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(np.asarray(y_true), np.asarray(y_pred), np.asarray(y_train))
    }

model_df = feature_df.dropna(subset=feature_cols + ["demand_clean"]).reset_index(drop=True)
train_mask = model_df["split"] == "train"
valid_mask = model_df["split"] == "valid"
test_mask = model_df["split"] == "test"

y_train = model_df.loc[train_mask, "demand_clean"].to_numpy()
y_valid = model_df.loc[valid_mask, "demand_clean"].to_numpy()

baseline_results = []
baseline_results.append(metric_row("persistence", y_valid, model_df.loc[valid_mask, "lag_1"], y_train))
baseline_results.append(metric_row("seasonal_naive", y_valid, model_df.loc[valid_mask, "lag_7"], y_train))
baseline_results.append(metric_row("moving_average_7", y_valid, model_df.loc[valid_mask, "roll_mean_7"], y_train))

pd.DataFrame(baseline_results).sort_values("MAE")

## 10. Candidate machine-learning models

We compare a small set of models. The goal is not to try every possible method. The goal is to create a fair comparison.

The preprocessing pipeline imputes missing feature values and scales numeric features when appropriate. The scaler must be fit only on the training portion inside the pipeline.

In [ ]:
X_train = model_df.loc[train_mask, feature_cols]
X_valid = model_df.loc[valid_mask, feature_cols]
X_test = model_df.loc[test_mask, feature_cols]
y_train = model_df.loc[train_mask, "demand_clean"]
y_valid = model_df.loc[valid_mask, "demand_clean"]
y_test = model_df.loc[test_mask, "demand_clean"]

models = {
    "ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),
    "random_forest_small": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=20,
            max_depth=8,
            min_samples_leaf=8,
            random_state=7339,
            n_jobs=1
        ))
    ]),
    "decision_tree_small": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeRegressor(
            max_depth=6,
            min_samples_leaf=12,
            random_state=7339
        ))
    ])
}

ml_results = []
valid_predictions = {}
for name, model in models.items():
    fitted = clone(model).fit(X_train, y_train)
    pred = fitted.predict(X_valid)
    valid_predictions[name] = pred
    ml_results.append(metric_row(name, y_valid, pred, y_train.to_numpy()))

results_valid = pd.concat([pd.DataFrame(baseline_results), pd.DataFrame(ml_results)], ignore_index=True)
results_valid.sort_values("MAE")

## 11. Visual validation comparison

Tables are not enough. Always plot the validation predictions against the actual series.

In [ ]:
best_name = results_valid.sort_values("MAE").iloc[0]["model"]
print("Best validation model by MAE:", best_name)

plt.figure(figsize=(10, 3.5))
plt.plot(model_df.loc[valid_mask, "ds"], y_valid, label="actual", linewidth=1.5)
plt.plot(model_df.loc[valid_mask, "ds"], model_df.loc[valid_mask, "lag_7"], label="seasonal naive", linewidth=1)
if best_name in valid_predictions:
    plt.plot(model_df.loc[valid_mask, "ds"], valid_predictions[best_name], label=best_name, linewidth=1)
else:
    baseline_map = {
        "persistence": model_df.loc[valid_mask, "lag_1"].to_numpy(),
        "seasonal_naive": model_df.loc[valid_mask, "lag_7"].to_numpy(),
        "moving_average_7": model_df.loc[valid_mask, "roll_mean_7"].to_numpy()
    }
    plt.plot(model_df.loc[valid_mask, "ds"], baseline_map[best_name], label=best_name, linewidth=1)
plt.title("Validation forecasts")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 12. Rolling-origin validation

A single validation split can be misleading. Rolling-origin validation repeats the forecasting exercise from multiple starting points.

For each origin:

1. train only on data before the origin,
2. predict the next block,
3. move the origin forward,
4. summarize errors across origins.

In [ ]:
def rolling_origin_evaluate(feature_df, feature_cols, models, target_col="demand_clean", initial_train_size=650, horizon=28, step=56):
    rows = []
    usable = feature_df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)
    y_all = usable[target_col].to_numpy()

    for origin in range(initial_train_size, len(usable) - horizon + 1, step):
        train_idx = np.arange(0, origin)
        eval_idx = np.arange(origin, origin + horizon)

        X_tr = usable.loc[train_idx, feature_cols]
        y_tr = usable.loc[train_idx, target_col]
        X_ev = usable.loc[eval_idx, feature_cols]
        y_ev = usable.loc[eval_idx, target_col].to_numpy()

        baseline_preds = {
            "persistence": usable.loc[eval_idx, "lag_1"].to_numpy(),
            "seasonal_naive": usable.loc[eval_idx, "lag_7"].to_numpy(),
            "moving_average_7": usable.loc[eval_idx, "roll_mean_7"].to_numpy()
        }
        for name, pred in baseline_preds.items():
            rows.append({
                "origin": origin,
                "model": name,
                "MAE": mean_absolute_error(y_ev, pred),
                "RMSE": rmse(y_ev, pred)
            })

        for name, model in models.items():
            fitted = clone(model).fit(X_tr, y_tr)
            pred = fitted.predict(X_ev)
            rows.append({
                "origin": origin,
                "model": name,
                "MAE": mean_absolute_error(y_ev, pred),
                "RMSE": rmse(y_ev, pred)
            })

    return pd.DataFrame(rows)

ro_results = rolling_origin_evaluate(feature_df, feature_cols, models)
ro_summary = ro_results.groupby("model", as_index=False).agg(
    MAE_mean=("MAE", "mean"),
    MAE_std=("MAE", "std"),
    RMSE_mean=("RMSE", "mean"),
    n_origins=("origin", "nunique")
).sort_values("MAE_mean")
ro_summary

In [ ]:
plt.figure(figsize=(8, 3.5))
for name, part in ro_results.groupby("model"):
    plt.plot(part["origin"], part["MAE"], marker="o", linewidth=1, label=name)
plt.title("Rolling-origin MAE by model")
plt.xlabel("origin row")
plt.ylabel("MAE over next 28 days")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 13. Final model choice and test evaluation

The test set should be used only after model selection. Here we choose the model with the best rolling-origin mean MAE.

In your final project, explain both the quantitative result and the modeling reason behind your choice.

In [ ]:
best_ro_name = ro_summary.iloc[0]["model"]
print("Selected model from rolling-origin validation:", best_ro_name)

# Train on train + validation, test once.
train_valid_mask = model_df["split"].isin(["train", "valid"])
X_train_valid = model_df.loc[train_valid_mask, feature_cols]
y_train_valid = model_df.loc[train_valid_mask, "demand_clean"]

baseline_test_preds = {
    "persistence": model_df.loc[test_mask, "lag_1"].to_numpy(),
    "seasonal_naive": model_df.loc[test_mask, "lag_7"].to_numpy(),
    "moving_average_7": model_df.loc[test_mask, "roll_mean_7"].to_numpy()
}

test_rows = []
for name, pred in baseline_test_preds.items():
    test_rows.append(metric_row(name, y_test, pred, y_train_valid.to_numpy()))

fitted_models = {}
for name, model in models.items():
    fitted = clone(model).fit(X_train_valid, y_train_valid)
    fitted_models[name] = fitted
    pred = fitted.predict(X_test)
    test_rows.append(metric_row(name, y_test, pred, y_train_valid.to_numpy()))

test_results = pd.DataFrame(test_rows).sort_values("MAE")
test_results

## 14. Residual diagnostics

A model can have a good average error but still fail systematically. Residual diagnostics help identify patterns not captured by the model.

In [ ]:
if best_ro_name in fitted_models:
    test_pred = fitted_models[best_ro_name].predict(X_test)
else:
    test_pred = baseline_test_preds[best_ro_name]

test_dates = model_df.loc[test_mask, "ds"]
resid = y_test.to_numpy() - test_pred

plt.figure(figsize=(10, 3.5))
plt.plot(test_dates, resid, linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title(f"Test residuals for selected model: {best_ro_name}")
plt.xlabel("date")
plt.ylabel("actual - forecast")
plt.show()

plt.figure(figsize=(5, 3.5))
plt.hist(resid, bins=20)
plt.title("Test residual histogram")
plt.xlabel("residual")
plt.ylabel("count")
plt.show()

In [ ]:
def sample_acf(x, max_lag=28):
    x = np.asarray(x, dtype=float)
    x = x - x.mean()
    denom = np.sum(x ** 2)
    vals = [1.0]
    for lag in range(1, max_lag + 1):
        vals.append(float(np.sum(x[:-lag] * x[lag:]) / denom))
    return np.array(vals)

acf_vals = sample_acf(resid, max_lag=28)
plt.figure(figsize=(8, 3.5))
plt.stem(np.arange(len(acf_vals)), acf_vals)
plt.axhline(0, linewidth=1)
plt.title("Residual sample ACF")
plt.xlabel("lag")
plt.ylabel("ACF")
plt.show()

## 15. Forecast intervals from validation residuals

A final project should distinguish point forecasts from uncertainty statements.

The simple interval below uses empirical validation residual quantiles. This is not the only way to create uncertainty intervals, but it is transparent and easy to explain.

In [ ]:
# Build validation residuals for the selected model using train-only fit.
if best_ro_name in models:
    selected_valid_model = clone(models[best_ro_name]).fit(X_train, y_train)
    valid_pred_selected = selected_valid_model.predict(X_valid)
else:
    valid_pred_selected = {
        "persistence": model_df.loc[valid_mask, "lag_1"].to_numpy(),
        "seasonal_naive": model_df.loc[valid_mask, "lag_7"].to_numpy(),
        "moving_average_7": model_df.loc[valid_mask, "roll_mean_7"].to_numpy()
    }[best_ro_name]

valid_resid = y_valid.to_numpy() - valid_pred_selected
lower_q, upper_q = np.quantile(valid_resid, [0.05, 0.95])

lower = test_pred + lower_q
upper = test_pred + upper_q
coverage = np.mean((y_test.to_numpy() >= lower) & (y_test.to_numpy() <= upper))
avg_width = np.mean(upper - lower)
print("Empirical 90% interval coverage on test set:", round(float(coverage), 3))
print("Average interval width:", round(float(avg_width), 3))

In [ ]:
plt.figure(figsize=(10, 3.5))
plt.plot(test_dates, y_test, label="actual", linewidth=1.5)
plt.plot(test_dates, test_pred, label="point forecast", linewidth=1)
plt.fill_between(test_dates, lower, upper, alpha=0.2, label="empirical 90% interval")
plt.title("Test forecasts with empirical residual intervals")
plt.xlabel("date")
plt.ylabel("demand")
plt.legend()
plt.show()

## 16. Multi-step direct forecasting matrix

Many final projects require several forecast horizons. A common approach is **direct forecasting**: train a separate target for each horizon.

For horizon `h`, the row at time `t` has target `y_{t+h}` and features available at time `t`.

In [ ]:
def make_direct_forecasting_table(feature_df, feature_cols, target_col="demand_clean", horizons=range(1, 15)):
    frames = []
    base = feature_df.copy()
    for h in horizons:
        temp = base[["ds", "split"] + feature_cols].copy()
        temp["horizon"] = h
        temp["target_future"] = base[target_col].shift(-h)
        temp["target_date"] = base["ds"].shift(-h)
        frames.append(temp)
    out = pd.concat(frames, ignore_index=True)
    out = out.dropna(subset=feature_cols + ["target_future"]).reset_index(drop=True)
    return out

direct_df = make_direct_forecasting_table(feature_df, feature_cols)
print(direct_df.shape)
direct_df.head()

In [ ]:
def evaluate_direct_ridge(direct_df, feature_cols):
    rows = []
    for h, part in direct_df.groupby("horizon"):
        train_part = part[part["split"].isin(["train", "valid"])]
        test_part = part[part["split"] == "test"]
        if len(test_part) == 0:
            continue
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=10.0))
        ])
        model.fit(train_part[feature_cols], train_part["target_future"])
        pred = model.predict(test_part[feature_cols])
        rows.append({
            "horizon": h,
            "MAE": mean_absolute_error(test_part["target_future"], pred),
            "RMSE": rmse(test_part["target_future"], pred),
            "n_test_rows": len(test_part)
        })
    return pd.DataFrame(rows)

horizon_results = evaluate_direct_ridge(direct_df, feature_cols)
horizon_results

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(horizon_results["horizon"], horizon_results["MAE"], marker="o")
plt.title("Direct ridge horizon-wise test MAE")
plt.xlabel("forecast horizon")
plt.ylabel("MAE")
plt.show()

## 17. A reusable final-project report table

A final report should not only state which model won. It should summarize the modeling contract, data, validation design, results, and limitations.

The next cell creates a compact project summary table that you can export or copy into your report.

In [ ]:
summary_table = pd.DataFrame({
    "item": [
        "target",
        "horizon",
        "information set",
        "validation design",
        "primary metric",
        "selected model",
        "best test MAE",
        "interval coverage",
        "main limitation"
    ],
    "project answer": [
        project_contract["target"],
        project_contract["forecast_horizon"],
        "; ".join(project_contract["information_set"]),
        project_contract["validation_design"],
        project_contract["primary_metric"],
        best_ro_name,
        round(float(test_results.iloc[0]["MAE"]), 3),
        round(float(coverage), 3),
        "simulated data; real data may contain structural breaks, data revisions, or unavailable exogenous variables"
    ]
})
summary_table

## 18. Model card template

A model card is a short document that makes the project auditable. Edit the text below for your final project.

In [ ]:
model_card = f'''
# Model Card: {project_contract['working_title']}

## Task
Predict {project_contract['target']} for horizon {project_contract['forecast_horizon']}.

## Data
The example data set has {len(df_raw)} daily observations from {df_raw['ds'].min().date()} to {df_raw['ds'].max().date()}.
Rows are ordered by time. Missing target values are filled for lag construction using forward fill and median fill.

## Information available at prediction time
{'; '.join(project_contract['information_set'])}

## Forbidden information
{'; '.join(project_contract['forbidden_information'])}

## Validation
{project_contract['validation_design']}.
The test set is used only after model selection.

## Baselines
{', '.join(project_contract['baselines'])}.

## Selected model
{best_ro_name}.

## Test performance
{test_results.to_string(index=False)}

## Uncertainty
Empirical 90% residual interval coverage on the test set: {coverage:.3f}.

## Limitations
This notebook uses simulated data. A real project must discuss data provenance, missingness, outliers, structural breaks, fairness or ethical concerns when relevant, and deployment risks.
'''

print(model_card)

## 19. Reproducibility checklist

Use the checklist below before submitting your final project.

In [ ]:
reproducibility_checklist = pd.DataFrame({
    "checkpoint": [
        "Project contract is stated before modeling",
        "Data source and time range are documented",
        "Target and prediction time are clearly defined",
        "Forbidden future information is listed",
        "Train/validation/test split is chronological",
        "Baselines are included",
        "Feature alignment tests are included",
        "Preprocessing is fit only on training data",
        "Rolling-origin validation is reported",
        "Test set is used once after selection",
        "Residual diagnostics are shown",
        "Forecast uncertainty is reported",
        "Limitations are discussed",
        "Code can be rerun from a clean runtime"
    ],
    "status": ["yes"] * 14,
    "evidence in this lab": [
        "Section 2",
        "Sections 3-4",
        "Section 2",
        "Section 2",
        "Section 6",
        "Section 9",
        "Section 8",
        "Sections 10 and 13",
        "Section 12",
        "Section 13",
        "Section 14",
        "Section 15",
        "Sections 17-18",
        "Notebook uses fixed seed and no external downloads"
    ]
})
reproducibility_checklist

## 20. AI-assisted project review prompts

Use AI as a critic, not as a substitute for validation. Copy one of the prompts below into an AI assistant and then verify every suggestion yourself.

In [ ]:
ai_review_prompts = [
    "I am forecasting a time-ordered target. Here is my project contract. Identify possible leakage risks and ask five clarification questions before suggesting models.",
    "Here are my feature definitions. For each feature, classify it as allowed, suspicious, or forbidden at prediction time. Explain your reasoning.",
    "Here are my validation and test results. Suggest residual diagnostics and robustness checks that would make the project more convincing.",
    "Here is my model card. Rewrite the limitations section to be more honest and specific without overstating the result.",
    "Here is my final-project abstract. Check whether the target, horizon, baseline, validation design, and main result are all clear."
]
for i, prompt in enumerate(ai_review_prompts, start=1):
    print(f"Prompt {i}: {prompt}")
    print()

## 21. Final project options

Here are several possible final-project directions. Your own project may be different.

1. **Forecasting project:** sales, demand, load, traffic, temperature, visits, or financial volatility.
2. **Classification project:** classify windows as normal/anomalous, regime A/regime B, or event/no event.
3. **Representation project:** learn embeddings of windows and cluster, retrieve, or visualize them.
4. **Spectral project:** identify periodic structure, filtering effects, or time-varying frequency behavior.
5. **Deep-learning project:** compare MLP, Conv1D, RNN/GRU/LSTM, or Transformer models using the same validation design.

A good final project is not the most complicated model. It is the project with the clearest question, cleanest validation, and most honest interpretation.

## 22. Exercises

1. Replace the simulated data with a real data set. Write a new project contract before modeling.
2. Add one new baseline. Examples: last-14-day average, trend extrapolation, or seasonal average by day of week.
3. Add one new feature and justify why it is available at prediction time.
4. Repeat rolling-origin validation with a different horizon and step size.
5. Compare the selected model against a neural network model from Labs 19-22.
6. Replace empirical residual intervals with split conformal intervals.
7. Write a one-page model card for your final project.
8. Create a five-slide presentation: question, data, method, validation, conclusion.

## 23. Mini-project: turn this notebook into your final project

Use this plan.

- **Step 1:** Replace Section 3 with your data-loading code.
- **Step 2:** Rewrite the project contract in Section 2.
- **Step 3:** Redo the data audit in Section 4.
- **Step 4:** Rewrite the feature factory in Section 7.
- **Step 5:** Add alignment tests in Section 8.
- **Step 6:** Choose at least two baselines in Section 9.
- **Step 7:** Compare at least two model classes in Sections 10-13.
- **Step 8:** Add residual diagnostics and uncertainty in Sections 14-15.
- **Step 9:** Write the model card in Section 18.
- **Step 10:** Prepare the final report and presentation.

Your final submission should be reproducible from a clean runtime.